# E2VID — Training Notebook

**Hybrid Vision · AMI 2026**

Train (or fine-tune) the **E2VID** recurrent UNet to reconstruct intensity frames from event-camera voxel grids.

| Setting | Value |
|---|---|
| Model | Recurrent UNet (Rebecq et al., CVPR 2019) |
| Dataset | FRED (Florence RGB-Event Drone) |
| Input | Voxel grid `(num_bins, H, W)` |
| Target | Grayscale intensity frame `(1, H, W)` in [0, 1] |
| Loss | λ·L1 + λ·(1−SSIM) + λ·TV |
| Optim | AdamW + Warmup-Cosine LR |

---

**Workflow**
1. Install dependencies
2. Clone repo & configure paths
3. Build dataset
4. Define loss & training loop
5. Run training
6. Visualise reconstructions

## 1 — Install dependencies

In [ ]:
# Run only if packages are not yet installed
# !pip install -q torch torchvision gdown h5py opencv-python-headless pyyaml

## 2 — Clone repo & set up paths

In [ ]:
import os, sys

# ── Google Colab ──────────────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
REPO = "/content/AMI-KK" if IN_COLAB else os.path.abspath("..")

if IN_COLAB and not os.path.exists(REPO):
    !git clone --depth 1 https://github.com/Dark-Fantasy-K/AMI-KK.git {REPO}

if REPO not in sys.path:
    sys.path.insert(0, REPO)

os.chdir(REPO)
print("Working directory:", os.getcwd())

## 3 — Configuration

Edit the paths and hyperparameters here.

In [ ]:
from pathlib import Path
import torch

# ── Paths ─────────────────────────────────────────────────────────────────
FRED_ROOT   = "/data/FRED"          # FRED dataset root
WEIGHTS     = None                  # Pretrained rpg_e2vid .pth.tar, or None to train from scratch
# WEIGHTS   = "models/reconstruction/e2vid/pretrained_weights/E2VID_lightweight.pth.tar"
SAVE_DIR    = Path("checkpoints/e2vid")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ── Data ──────────────────────────────────────────────────────────────────
NUM_BINS    = 5       # Voxel grid time bins (must match pretrained model if fine-tuning)
WINDOW_SIZE = 10      # Consecutive frames per training item
INPUT_SIZE  = None    # Resize to N×N; None = keep original resolution
NUM_WORKERS = 2

# ── Training ──────────────────────────────────────────────────────────────
EPOCHS      = 50
BATCH_SIZE  = 2
LR          = 1e-4
WEIGHT_DECAY= 1e-5
WARMUP      = 3       # LR warmup epochs
TBPTT_STEPS = 5       # Truncated BPTT chunk size
GRAD_CLIP   = 1.0
USE_AMP     = True    # Mixed-precision (CUDA only)

# ── Loss weights ──────────────────────────────────────────────────────────
LAMBDA_L1   = 1.0
LAMBDA_SSIM = 0.5
LAMBDA_TV   = 1e-4

# ── Device ────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

## 4 — Dataset

In [ ]:
from __future__ import annotations
import random
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

from models.reconstruction.utils import (
    load_events_for_sequence,
    load_frame_timestamps,
    events_to_voxel,
)


class E2VIDTrainDataset(Dataset):
    """
    Sliding-window dataset over FRED sequences.

    Returns:
        voxels  : Tensor (K, num_bins, H, W)  — event voxel grids
        targets : Tensor (K, 1, H, W)         — grayscale frames in [0, 1]
    """

    def __init__(
        self,
        fred_root,
        split="train",
        num_bins=5,
        window_size=10,
        input_size=None,
        augment=True,
    ):
        self.num_bins    = num_bins
        self.window_size = window_size
        self.input_size  = input_size
        self.augment     = augment and (split == "train")

        seq_root = Path(fred_root) / split / "sequences"
        if not seq_root.exists():
            raise FileNotFoundError(f"Split not found: {seq_root}")

        self.windows: list[tuple] = []
        stride = max(1, window_size // 2)

        for seq_dir in sorted(p for p in seq_root.iterdir() if p.is_dir()):
            frames = sorted((seq_dir / "rgb").glob("*.jpg")) + \
                     sorted((seq_dir / "rgb").glob("*.png"))
            n = len(frames)
            if n < window_size:
                continue
            for start in range(0, n - window_size + 1, stride):
                self.windows.append((seq_dir, start))

        if not self.windows:
            raise RuntimeError("No windows found — check fred_root and window_size.")
        print(f"[{split}] {len(self.windows)} windows from {seq_root}")

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        seq_dir, start = self.windows[idx]

        events     = load_events_for_sequence(seq_dir)
        rgb_paths  = sorted((seq_dir / "rgb").glob("*.jpg")) + \
                     sorted((seq_dir / "rgb").glob("*.png"))
        timestamps = load_frame_timestamps(seq_dir)
        if len(timestamps) != len(rgb_paths):
            timestamps = np.linspace(0.0, (len(rgb_paths) - 1) / 30.0, len(rgb_paths))

        ref = cv2.imread(str(rgb_paths[start]))
        H_src, W_src = ref.shape[:2]
        flip = self.augment and random.random() < 0.5

        voxels, targets = [], []
        for fi in range(start, start + self.window_size):
            img_bgr = cv2.imread(str(rgb_paths[fi]))
            if img_bgr is None:
                img_bgr = np.zeros((H_src, W_src, 3), dtype=np.uint8)
            gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)

            t_end   = timestamps[fi]
            t_start = timestamps[fi - 1] if fi > 0 else t_end - 1.0 / 30.0
            voxel   = events_to_voxel(events, H_src, W_src, self.num_bins, t_start, t_end)

            if self.input_size:
                sz = self.input_size
                gray  = cv2.resize(gray, (sz, sz))
                voxel = np.stack([cv2.resize(voxel[b], (sz, sz)) for b in range(self.num_bins)])

            if flip:
                gray  = gray[:,  ::-1].copy()
                voxel = voxel[:, :, ::-1].copy()

            voxels.append(torch.from_numpy(voxel.copy()))
            targets.append(torch.from_numpy(gray.astype(np.float32) / 255.0).unsqueeze(0))

        return torch.stack(voxels), torch.stack(targets)

In [ ]:
train_ds = E2VIDTrainDataset(
    FRED_ROOT, split="train",
    num_bins=NUM_BINS, window_size=WINDOW_SIZE,
    input_size=INPUT_SIZE, augment=True,
)
val_ds = E2VIDTrainDataset(
    FRED_ROOT, split="test",
    num_bins=NUM_BINS, window_size=WINDOW_SIZE,
    input_size=INPUT_SIZE, augment=False,
)
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)
print(f"Train batches: {len(train_loader)}  |  Val batches: {len(val_loader)}")

# Sanity-check one item
v, t = train_ds[0]
print(f"Voxel shape : {v.shape}  dtype={v.dtype}  range=[{v.min():.2f}, {v.max():.2f}]")
print(f"Target shape: {t.shape}  dtype={t.dtype}  range=[{t.min():.2f}, {t.max():.2f}]")

## 5 — Model

In [ ]:
from models.reconstruction.e2vid.model import E2VID, load_e2vid

if WEIGHTS:
    model = load_e2vid(WEIGHTS, DEVICE)
    model.train()
    print(f"Fine-tuning from: {WEIGHTS}")
else:
    model = E2VID(num_bins=NUM_BINS).to(DEVICE)
    print("Training from scratch")

n_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {n_params:,}  (~{n_params/1e6:.1f} M)")

## 6 — Loss functions

In [ ]:
import torch.nn as nn
import torch.nn.functional as F


def _ssim(pred, target, k=11):
    C1, C2 = 0.01 ** 2, 0.03 ** 2
    pad = k // 2
    def pool(x):
        return F.avg_pool2d(x, k, stride=1, padding=pad)
    mu_p, mu_t = pool(pred), pool(target)
    sp  = pool(pred   * pred)   - mu_p * mu_p
    st  = pool(target * target) - mu_t * mu_t
    spt = pool(pred   * target) - mu_p * mu_t
    num = (2 * mu_p * mu_t + C1) * (2 * spt + C2)
    den = (mu_p**2 + mu_t**2 + C1) * (sp + st + C2)
    return (num / den).mean()


def _tv(x):
    return (
        (x[:, :, 1:, :] - x[:, :, :-1, :]).abs().mean()
        + (x[:, :, :, 1:] - x[:, :, :, :-1]).abs().mean()
    )


class E2VIDLoss(nn.Module):
    def __init__(self, w_l1=1.0, w_ssim=0.5, w_tv=1e-4):
        super().__init__()
        self.w_l1, self.w_ssim, self.w_tv = w_l1, w_ssim, w_tv

    def forward(self, pred, target):
        l1   = F.l1_loss(pred, target)
        ssim = 1.0 - _ssim(pred, target)
        tv   = _tv(pred)
        loss = self.w_l1 * l1 + self.w_ssim * ssim + self.w_tv * tv
        return loss, {"l1": l1.item(), "ssim": ssim.item(), "tv": tv.item()}


criterion = E2VIDLoss(LAMBDA_L1, LAMBDA_SSIM, LAMBDA_TV)
print("Loss: E2VIDLoss(L1 + SSIM + TV) ready")

## 7 — Optimizer & scheduler

In [ ]:
import math

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return (epoch + 1) / max(WARMUP, 1)
    t = (epoch - WARMUP) / max(EPOCHS - WARMUP, 1)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * t))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler    = torch.cuda.amp.GradScaler() if (USE_AMP and DEVICE.type == "cuda") else None

print(f"Optimizer : AdamW  lr={LR}  wd={WEIGHT_DECAY}")
print(f"Scheduler : warmup-{WARMUP} + cosine  AMP={'on' if scaler else 'off'}")

## 8 — Training & validation helpers

In [ ]:
def detach_states(model):
    """Stop gradients flowing through LSTM states between TBPTT chunks."""
    for i, s in enumerate(model.unet.states):
        if s is not None:
            h, c = s
            model.unet.states[i] = (h.detach(), c.detach())


def train_one_epoch(model, loader, optimizer, criterion, device,
                    tbptt_steps=5, grad_clip=1.0, scaler=None):
    model.train()
    totals = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0

    for voxels, targets in loader:
        voxels  = voxels.to(device, non_blocking=True)   # (B, K, bins, H, W)
        targets = targets.to(device, non_blocking=True)  # (B, K, 1, H, W)
        K = voxels.shape[1]
        model.unet.reset_states()

        for t0 in range(0, K, tbptt_steps):
            t1 = min(t0 + tbptt_steps, K)
            optimizer.zero_grad()
            chunk_loss = torch.tensor(0.0, device=device)

            for t in range(t0, t1):
                with torch.autocast(device_type=device.type, enabled=(scaler is not None)):
                    pred = model(voxels[:, t])
                    loss, parts = criterion(pred, targets[:, t])
                chunk_loss = chunk_loss + loss

            if scaler:
                scaler.scale(chunk_loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                chunk_loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            detach_states(model)
            totals["loss"] += chunk_loss.item() / (t1 - t0)
            totals["l1"]   += parts["l1"]
            totals["ssim"] += parts["ssim"]
            n += 1

    return {k: v / max(n, 1) for k, v in totals.items()}


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    totals = {"loss": 0.0, "l1": 0.0, "ssim": 0.0}
    n = 0
    for voxels, targets in loader:
        voxels  = voxels.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        model.unet.reset_states()
        for t in range(voxels.shape[1]):
            pred = model(voxels[:, t])
            loss, parts = criterion(pred, targets[:, t])
            totals["loss"] += loss.item()
            totals["l1"]   += parts["l1"]
            totals["ssim"] += parts["ssim"]
            n += 1
    return {k: v / max(n, 1) for k, v in totals.items()}

print("Helpers defined.")

## 9 — Run training

Metrics are logged per epoch. Checkpoints are saved to `SAVE_DIR/last.pth` and `best.pth`.

In [ ]:
import time

history = {"train_loss": [], "val_loss": [], "lr": []}
best_val = float("inf")

for epoch in range(EPOCHS):
    t0 = time.time()

    tr = train_one_epoch(
        model, train_loader, optimizer, criterion, DEVICE,
        tbptt_steps=TBPTT_STEPS, grad_clip=GRAD_CLIP, scaler=scaler,
    )
    vl = validate(model, val_loader, criterion, DEVICE)
    scheduler.step()
    elapsed = time.time() - t0
    current_lr = scheduler.get_last_lr()[0]

    history["train_loss"].append(tr["loss"])
    history["val_loss"].append(vl["loss"])
    history["lr"].append(current_lr)

    print(
        f"Epoch {epoch+1:3d}/{EPOCHS} | {elapsed:4.0f}s | lr={current_lr:.2e} | "
        f"train: loss={tr['loss']:.4f} L1={tr['l1']:.4f} SSIM={tr['ssim']:.4f} | "
        f"val: loss={vl['loss']:.4f} L1={vl['l1']:.4f}"
    )

    ckpt = {
        "epoch": epoch,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "best_val_loss": best_val,
    }
    torch.save(ckpt, SAVE_DIR / "last.pth")

    if vl["loss"] < best_val:
        best_val = vl["loss"]
        torch.save(ckpt, SAVE_DIR / "best.pth")
        print(f"  ✓ New best val loss: {best_val:.4f}")

print(f"\nTraining complete. Best val loss: {best_val:.4f}")

## 10 — Loss curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"],   label="val")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].set_title("E2VID Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history["lr"])
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Learning Rate")
axes[1].set_title("LR Schedule (warmup + cosine)")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(SAVE_DIR / "loss_curves.png", dpi=120)
plt.show()

## 11 — Visualise reconstructions from validation set

In [ ]:
import matplotlib.pyplot as plt

model.eval()
model.unet.reset_states()

voxels, targets = next(iter(val_loader))
voxels  = voxels.to(DEVICE)
targets = targets.to(DEVICE)

K       = voxels.shape[1]
N_SHOW  = min(6, K)
step    = max(1, K // N_SHOW)
frames  = list(range(0, K, step))[:N_SHOW]

preds = []
with torch.no_grad():
    for t in range(K):
        p = model(voxels[:, t])
        if t in frames:
            preds.append(p[0, 0].cpu().numpy())

fig, axes = plt.subplots(2, N_SHOW, figsize=(3 * N_SHOW, 6))
for i, (fi, pred) in enumerate(zip(frames, preds)):
    gt = targets[0, fi, 0].cpu().numpy()
    axes[0, i].imshow(gt,   cmap="gray", vmin=0, vmax=1)
    axes[0, i].set_title(f"GT  frame {fi}")
    axes[0, i].axis("off")
    axes[1, i].imshow(pred, cmap="gray", vmin=0, vmax=1)
    axes[1, i].set_title(f"Pred frame {fi}")
    axes[1, i].axis("off")

axes[0, 0].set_ylabel("Ground truth", fontsize=10)
axes[1, 0].set_ylabel("E2VID output", fontsize=10)
plt.suptitle("E2VID Reconstructions (validation)", fontsize=13)
plt.tight_layout()
plt.savefig(SAVE_DIR / "sample_reconstructions.png", dpi=120)
plt.show()

## 12 — Load best checkpoint & export

In [ ]:
# Load the best checkpoint saved during training
best_ckpt_path = SAVE_DIR / "best.pth"
ckpt = torch.load(best_ckpt_path, map_location=DEVICE)
model.load_state_dict(ckpt["model"])
print(f"Loaded best checkpoint from epoch {ckpt['epoch'] + 1}")
print(f"Best val loss: {ckpt['best_val_loss']:.4f}")

In [ ]:
# Optional: export just the model weights in rpg_e2vid-compatible format
export_path = SAVE_DIR / "e2vid_trained.pth"
torch.save({"state_dict": model.state_dict()}, export_path)
print(f"Model weights exported to: {export_path}")
print("Use with: load_e2vid(export_path, device)")